# Importation

In [1]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

from src.telechargement import RACINE_PROJET, charger_parametres, telecharger_sources
from src.controles import (
    AnomalieBloquante,
    Volumetrie,
    compter_cles_orphelines,
    compter_doublons,
    compter_hors_bornes,
    compter_incoherence_nb_sinistres,
    compter_lignes_dupliquees,
    compter_valeurs_egales,
    verifier_niveau,
)
from src.jointure import (
    agreger_sinistres,
    calculer_ecart_nb_sinistres,
    detailler_ecart_nb_sinistres,
    isoler_orphelins,
    joindre_contrats_sinistres,
)

# Paramètres

In [2]:
os.environ["JAVA_HOME"] = os.path.join(sys.prefix, "Library")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("data_analysis")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

parametres = charger_parametres()


# Chargement des données

In [3]:
# Garantit que les CSV existent (sklearn met en cache les données
# OpenML : relancer cette cellule ne retélécharge pas depuis internet).
telecharger_sources(parametres)

schema_contrats = StructType(
    [
        StructField("IDpol", DoubleType(), nullable=False),
        StructField("ClaimNb", IntegerType(), nullable=False),
        StructField("Exposure", DoubleType(), nullable=False),
        StructField("Area", StringType(), nullable=True),
        StructField("VehPower", IntegerType(), nullable=False),
        StructField("VehAge", IntegerType(), nullable=False),
        StructField("DrivAge", IntegerType(), nullable=False),
        StructField("BonusMalus", IntegerType(), nullable=False),
        StructField("VehBrand", StringType(), nullable=True),
        StructField("VehGas", StringType(), nullable=True),
        StructField("Density", IntegerType(), nullable=False),
        StructField("Region", StringType(), nullable=True),
    ]
)
schema_sinistres = StructType(
    [
        StructField("IDpol", DoubleType(), nullable=False),
        StructField("ClaimAmount", DoubleType(), nullable=False),
    ]
)

chemin_contrats = RACINE_PROJET / parametres["sources"]["contrats"]["fichier"]
chemin_sinistres = RACINE_PROJET / parametres["sources"]["sinistres"]["fichier"]

df_contrats = spark.read.csv(str(chemin_contrats), header=True, schema=schema_contrats)
df_sinistres = spark.read.csv(str(chemin_sinistres), header=True, schema=schema_sinistres)




In [4]:
# IDpol est la clé de jointure (config/parametres.yaml: jointure.cle).
# En assurance, un identifiant de police est une clé, pas une grandeur
# numérique : on le type en string. Le CSV des contrats écrit "1.0"
# (artefact du jeu de données d'origine) ; on repasse par un entier pour
# obtenir "1" plutôt que "1.0" avant de convertir en texte.
df_contrats = df_contrats.withColumn("IDpol", F.col("IDpol").cast("long").cast("string"))
df_sinistres = df_sinistres.withColumn("IDpol", F.col("IDpol").cast("long").cast("string"))

df_contrats.show(5)
df_sinistres.show(5)

+-----+-------+--------+----+--------+------+-------+----------+--------+---------+-------+------+
|IDpol|ClaimNb|Exposure|Area|VehPower|VehAge|DrivAge|BonusMalus|VehBrand|   VehGas|Density|Region|
+-----+-------+--------+----+--------+------+-------+----------+--------+---------+-------+------+
|    1|      1|     0.1|   D|       5|     0|     55|        50|     B12|'Regular'|   1217|   R82|
|    3|      1|    0.77|   D|       5|     0|     55|        50|     B12|'Regular'|   1217|   R82|
|    5|      1|    0.75|   B|       6|     2|     52|        50|     B12| 'Diesel'|     54|   R22|
|   10|      1|    0.09|   B|       7|     0|     46|        50|     B12| 'Diesel'|     76|   R72|
|   11|      1|    0.84|   B|       7|     0|     46|        50|     B12| 'Diesel'|     76|   R72|
+-----+-------+--------+----+--------+------+-------+----------+--------+---------+-------+------+
only showing top 5 rows
+-------+-----------+
|  IDpol|ClaimAmount|
+-------+-----------+
|   1552|      995.

# Bases Contrats

In [5]:
df_contrats.printSchema()

root
 |-- IDpol: string (nullable = true)
 |-- ClaimNb: integer (nullable = true)
 |-- Exposure: double (nullable = true)
 |-- Area: string (nullable = true)
 |-- VehPower: integer (nullable = true)
 |-- VehAge: integer (nullable = true)
 |-- DrivAge: integer (nullable = true)
 |-- BonusMalus: integer (nullable = true)
 |-- VehBrand: string (nullable = true)
 |-- VehGas: string (nullable = true)
 |-- Density: integer (nullable = true)
 |-- Region: string (nullable = true)



In [6]:
df_contrats.describe("Exposure", "ClaimNb", "BonusMalus", "VehPower", "DrivAge", "VehAge").show()

+-------+-------------------+-------------------+------------------+------------------+------------------+------------------+
|summary|           Exposure|            ClaimNb|        BonusMalus|          VehPower|           DrivAge|            VehAge|
+-------+-------------------+-------------------+------------------+------------------+------------------+------------------+
|  count|             678013|             678013|            678013|            678013|            678013|            678013|
|   mean| 0.5287501057694226|0.05324676665491664| 59.76150162312522| 6.454631400872844| 45.49912169825652| 7.044264637993667|
| stddev|0.36444154632153325|0.24011733044675934|15.636657659861433|2.0509056977051925|14.137444074314976|5.6662315799724485|
|    min| 0.0027322404371584|                  0|                50|                 4|                18|                 0|
|    max|               2.01|                 16|               230|                15|               100|            

In [7]:

df_contrats.select("Exposure", "ClaimNb", "BonusMalus", "VehPower", "DrivAge", "VehAge").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

+-------+-------------------+-------------------+------------------+------------------+------------------+------------------+
|summary|           Exposure|            ClaimNb|        BonusMalus|          VehPower|           DrivAge|            VehAge|
+-------+-------------------+-------------------+------------------+------------------+------------------+------------------+
|  count|             678013|             678013|            678013|            678013|            678013|            678013|
|   mean| 0.5287501057694226|0.05324676665491664| 59.76150162312522| 6.454631400872844| 45.49912169825652| 7.044264637993667|
| stddev|0.36444154632153325|0.24011733044675934|15.636657659861433|2.0509056977051925|14.137444074314976|5.6662315799724485|
|    min| 0.0027322404371584|                  0|                50|                 4|                18|                 0|
|    25%|               0.18|                  0|                50|                 5|                34|            

- **Exposition** : Va de 1 jour à 2 ans
- **ClaimNb** : De 0 à 16 sinistres
- **BonusMalus** (Coef de reduction /Majoration) : De 50 - 230


In [8]:
#Unicité de la clé IDpol (pas de doublons présents dans le DataFrame   )
df_contrats.select("IDpol").distinct().count() == df_contrats.count()

True

In [9]:
#Valeur manquantes pour chaque colonne (nombre de valeurs nulles ou vides présentes dans le DataFrame pour chaque colonne)
# On caste en string avant de comparer à "" : comparer directement une
# colonne numérique à "" force Spark à convertir "" en nombre, ce qui
# échoue (CAST_INVALID_INPUT). Convertir la colonne en texte est toujours
# valide, quel que soit son type d'origine.
df_contrats.select([F.count(F.when(F.col(c).isNull() | (F.col(c).cast("string") == ""), c)).alias(c) for c in df_contrats.columns]).show()


+-----+-------+--------+----+--------+------+-------+----------+--------+------+-------+------+
|IDpol|ClaimNb|Exposure|Area|VehPower|VehAge|DrivAge|BonusMalus|VehBrand|VehGas|Density|Region|
+-----+-------+--------+----+--------+------+-------+----------+--------+------+-------+------+
|    0|      0|       0|   0|       0|     0|      0|         0|       0|     0|      0|     0|
+-----+-------+--------+----+--------+------+-------+----------+--------+------+-------+------+



# Base sinistres

In [10]:


df_sinistres.printSchema()

root
 |-- IDpol: string (nullable = true)
 |-- ClaimAmount: double (nullable = true)



In [11]:

df_sinistres.select("ClaimAmount").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

+-------+-----------------+
|summary|      ClaimAmount|
+-------+-----------------+
|  count|            26639|
|   mean|2278.536381996256|
| stddev|  29297.481336158|
|    min|              1.0|
|    25%|           686.26|
|    50%|           1172.0|
|    75%|          1228.08|
|    max|       4075400.56|
+-------+-----------------+



In [12]:
#Unicité de la clé IDpol dans le DataFrame des sinistres
#pas uniques un contrats à plusieurs sinistres
df_sinistres.select("IDpol").distinct().count() 

24950

In [13]:
#Nombre total de lignes dans le DataFrame des sinistres
df_sinistres.count()

26639

In [14]:
df_sinistres.groupby("IDpol").count().filter("count > 20").show()

+-------+-----+
|  IDpol|count|
+-------+-----+
|2220367|   24|
|2277846|   23|
|2262511|   66|
|2227533|   25|
|2286775|   21|
|2282134|   36|
+-------+-----+



In [15]:
#2220367
df_contrats.filter("IDpol = 2262511").show(30)

+-----+-------+--------+----+--------+------+-------+----------+--------+------+-------+------+
|IDpol|ClaimNb|Exposure|Area|VehPower|VehAge|DrivAge|BonusMalus|VehBrand|VehGas|Density|Region|
+-----+-------+--------+----+--------+------+-------+----------+--------+------+-------+------+
+-----+-------+--------+----+--------+------+-------+----------+--------+------+-------+------+



In [16]:
#2220367
df_sinistres.filter("IDpol = 2262511").show(30)

+-------+-----------+
|  IDpol|ClaimAmount|
+-------+-----------+
|2262511|    2341.37|
|2262511|     1500.0|
|2262511|     135.97|
|2262511|     1500.0|
|2262511|    19461.0|
|2262511|      230.0|
|2262511|     1500.0|
|2262511|    10000.0|
|2262511|     2000.0|
|2262511|    2898.97|
|2262511|     109.92|
|2262511|     3000.0|
|2262511|    2129.87|
|2262511|     1500.0|
|2262511|    1414.22|
|2262511|    3105.56|
|2262511|     5000.0|
|2262511|    10000.0|
|2262511|     1500.0|
|2262511|     8000.0|
|2262511|     4000.0|
|2262511|     1500.0|
|2262511|    2001.12|
|2262511|    1483.44|
|2262511|     346.54|
|2262511|     1202.3|
|2262511|    1012.46|
|2262511|    2300.12|
|2262511|    1625.71|
|2262511|    2731.39|
+-------+-----------+
only showing top 30 rows


- Certains contrat on plusieurs sinistres car plusieurs ligne mais leur nombre de sinistres ne correspond pas au nombre de sinistres dans la base contrat

In [17]:
#Creation de la nouvelle base sinistre sans doublons sur IDpol et une colonne nb_sinistres
df_sinistres_unique = df_sinistres.groupby("IDpol").count().withColumnRenamed("count", "nb_sinistres")
df_sinistres_unique.show(30)

+-------+------------+
|  IDpol|nb_sinistres|
+-------+------------+
|5077810|           1|
|5040782|           1|
|5000906|           1|
|5064188|           1|
|5090453|           1|
|6010292|           2|
|6043211|           1|
|1013585|           1|
|6092582|           1|
|6079615|           1|
|2012363|           2|
|2017973|           1|
|   9993|           1|
|1005483|           1|
| 100227|           1|
| 100140|           1|
|  23459|           1|
|  59722|           1|
|  64793|           1|
| 129005|           1|
|  77371|           1|
|  63738|           1|
|1123761|           1|
|1035341|           1|
|1041410|           1|
|1075545|           1|
|1092176|           1|
|1044108|           1|
|1105788|           1|
|1091675|           1|
+-------+------------+
only showing top 30 rows


In [18]:
df_sinistres.count()

26639

In [19]:
df_sinistres_unique.count()

24950

In [20]:
df_sinistres_unique.filter(df_sinistres_unique["nb_sinistres"] > 16).show(30 )

+-------+------------+
|  IDpol|nb_sinistres|
+-------+------------+
|2220367|          24|
|2277846|          23|
|2262511|          66|
|2227533|          25|
|2286775|          21|
|2282134|          36|
+-------+------------+



## Verifications de plausibilite sur BonusMalus

Ces 3 controles ne font pas partie du perimetre officiel du cahier des charges. 
Ce sont des verifications exploratoires ajoutees
pour comprendre la qualite de la colonne `BonusMalus`. Les seuils viennent de `config/parametres.yaml` (`controles.bonus_malus`)

In [21]:
bm = parametres["controles"]["bonus_malus"]

# 1. Hors bornes legales (bloquant) : coefficient en dehors de [borne_min, borne_max]
hors_bornes = df_contrats.filter(
    (F.col("BonusMalus") < bm["borne_min"]) | (F.col("BonusMalus") > bm["borne_max"])
).count()

# 2. Non multiple du palier (alerte) : le CRM evolue normalement par paliers de 5
pas_non_standard = df_contrats.filter(F.col("BonusMalus") % bm["pas_palier"] != 0).count()

# 3. Jeune conducteur deja au plancher (alerte) : il faut ~13 ans sans sinistre
# pour atteindre le plancher ; improbable avant l'age defini dans le YAML
jeune_au_plancher = df_contrats.filter(
    (F.col("BonusMalus") == bm["borne_min"]) & (F.col("DrivAge") < bm["age_min_plancher"])
).count()

print(f"Hors bornes [{bm['borne_min']}, {bm['borne_max']}] : {hors_bornes}")
print(f"Non multiples de {bm['pas_palier']} : {pas_non_standard}")
print(f"Plancher avant {bm['age_min_plancher']} ans : {jeune_au_plancher}")

Hors bornes [50, 350] : 0
Non multiples de 5 : 177414
Plancher avant 25 ans : 972


In [22]:
df_contrats.filter(
    (F.col("BonusMalus") == bm["borne_min"]) & (F.col("DrivAge") < bm["age_min_plancher"])
).show()

+-----+-------+--------+----+--------+------+-------+----------+--------+---------+-------+------+
|IDpol|ClaimNb|Exposure|Area|VehPower|VehAge|DrivAge|BonusMalus|VehBrand|   VehGas|Density|Region|
+-----+-------+--------+----+--------+------+-------+----------+--------+---------+-------+------+
|17993|      1|     1.0|   C|       7|    17|     22|        50|      B2|'Regular'|    155|   R24|
|18671|      1|    0.07|   D|       7|     3|     22|        50|      B2| 'Diesel'|   1368|   R11|
|18672|      1|    0.21|   D|       7|     3|     22|        50|      B2| 'Diesel'|   1368|   R11|
|21190|      2|     1.0|   C|       5|     4|     22|        50|      B4|'Regular'|    287|   R53|
|22074|      1|     1.0|   A|       4|     1|     24|        50|      B6|'Regular'|     48|   R24|
|22215|      1|    0.94|   D|       4|    14|     20|        50|      B1|'Regular'|   1313|   R24|
|24291|      1|     1.0|   E|       4|     7|     22|        50|      B1|'Regular'|   3866|   R24|
|24393|   

## Contrôles officiels du cahier des charges

Ces 5 contrôles sont ceux listés dans le périmètre du cahier des charges
(section 3) : doublons, clés orphelines, cohérence des nombres de sinistres,
bornes de l'exposition. Les fonctions viennent de `src/controles.py` (testées
dans `tests/test_controles.py`, ENF3) ; les seuils et niveaux viennent de
`config/parametres.yaml` (ENF1).

In [23]:
niveaux = parametres["controles"]["niveaux"]

resultats = {
    "doublons_contrats": compter_doublons(df_contrats, "IDpol"),
    "doublons_sinistres": compter_lignes_dupliquees(df_sinistres),
    "cles_orphelines": compter_cles_orphelines(df_sinistres, df_contrats, "IDpol"),
    "coherence_nb_sinistres": compter_incoherence_nb_sinistres(
        df_contrats, df_sinistres, "IDpol", "ClaimNb"
    ),
    "exposition_hors_bornes": compter_hors_bornes(
        df_contrats,
        "Exposure",
        parametres["controles"]["exposition_min"],
        parametres["controles"]["exposition_max"],
    ),
}

for nom, nb_anomalies in resultats.items():
    print(f"[{niveaux[nom].upper():8}] {nom:28} : {nb_anomalies} anomalie(s)")

[BLOQUANT] doublons_contrats            : 0 anomalie(s)
[ALERTE  ] doublons_sinistres           : 255 anomalie(s)
[ALERTE  ] cles_orphelines              : 195 anomalie(s)
[ALERTE  ] coherence_nb_sinistres       : 9117 anomalie(s)
[ALERTE  ] exposition_hors_bornes       : 1224 anomalie(s)


- Les doublons sur les sinistres (255 lignes) ne sont pas simplement « un contrat avec plusieurs sinistres » : ce contrôle détecte des lignes strictement identiques (même IDpol ET même ClaimAmount au centime près). Un contrat avec plusieurs sinistres de montants différents n'est jamais compté ici. Ça peut venir d'un vrai doublon d'enregistrement, ou d'un paiement fractionné en plusieurs tranches du même montant — à investiguer. Il reste aussi à comprendre pourquoi le nombre de sinistres côté contrat et côté sinistres sont parfois différents (couvert par coherence_nb_sinistres).
- On a de vrai cas orphelin, certain contrat qui on des sinistres mais ne se retrouve pas dans la base des contrats, ceci peut etre du au fait que le contrat peut etre rentré tardivment ou juste deja résilie 
- Exposition sup à 1 n'est pas forcément une anomalies car on ne connais pas exactement sur quelle période les données sont récupérés. Mais pour effectuer des models de tarification en introduisant la variable année la contrinte de expo inf ou egale à 1  est necessaire mais pour ce data set je ne suis pas tres sur a moins que on est mentionnée que l'expo est annuelle et que j'i pas vu


## Contrôles exploratoires (hors périmètre officiel)

Ces contrôles ne sont pas demandés par le cahier des charges : ce sont des
vérifications de plausibilité ajoutées pendant l'exploration (comme pour
`BonusMalus` plus haut). Mêmes fonctions réutilisées depuis `src/controles.py`.

In [24]:
veh_age = parametres["controles"]["veh_age"]
driv_age = parametres["controles"]["driv_age"]
claim_nb = parametres["controles"]["claim_nb"]
claim_amount = parametres["controles"]["claim_amount"]

resultats_exploratoires = {
    "veh_age_hors_bornes": (
        compter_hors_bornes(df_contrats, "VehAge", borne_max=veh_age["borne_max"]),
        veh_age["niveau"],
    ),
    "driv_age_hors_bornes": (
        compter_hors_bornes(df_contrats, "DrivAge", driv_age["borne_min"], driv_age["borne_max"]),
        driv_age["niveau"],
    ),
    # un nombre de sinistres negatif est impossible ; pas de borne haute : un
    # contrat peut legitimement accumuler plus de 10 sinistres sur sa duree de vie
    "claim_nb_negatif": (
        compter_hors_bornes(df_contrats, "ClaimNb", borne_min=claim_nb["borne_min"]),
        claim_nb["niveau"],
    ),
    # un montant negatif n'est PAS une anomalie ici : il peut correspondre a
    # un recours/subrogation (l'assureur recupere de l'argent aupres d'un tiers)
    "claim_amount_nul": (
        compter_valeurs_egales(df_sinistres, "ClaimAmount", claim_amount["valeur_suspecte"]),
        claim_amount["niveau"],
    ),
}

for nom, (nb_anomalies, niveau) in resultats_exploratoires.items():
    print(f"[{niveau.upper():8}] {nom:28} : {nb_anomalies} anomalie(s)")

[ALERTE  ] veh_age_hors_bornes          : 0 anomalie(s)
[ALERTE  ] driv_age_hors_bornes         : 0 anomalie(s)
[ALERTE  ] claim_nb_negatif             : 0 anomalie(s)
[ALERTE  ] claim_amount_nul             : 0 anomalie(s)


## Jointure contrats x sinistres

Décisions prises avec Laurice pour cette étape (cf. `config/parametres.yaml`,
`jointure.type: left`) :

- LEFT JOIN : on garde tous les contrats (table de référence), les sinistres
  sont agrégés par `IDpol` (clé rendue unique) avant la jointure.
- Les lignes de sinistres strictement identiques (255 cas) ne sont pas
  filtrées : elles comptent comme de vrais sinistres (l'explication peut être
  légitime, ex. paiement fractionné).
- Les sinistres orphelins (sans contrat) sont isolés dans une base à part
  (`sorties.sinistres_orphelins`), pas supprimés : réutilisables plus tard
  pour le calage tarifaire.
- `ClaimNb` (déclaré) est conservé à côté de `nb_sinistres_reel` (recalculé) :
  l'écart n'est pas corrigé, seulement mesuré (nombre de contrats + montant
  concerné).

In [25]:
cle = parametres["jointure"]["cle"]
niveaux = parametres["controles"]["niveaux"]

volumetrie = Volumetrie()
volumetrie.enregistrer("contrats_bruts", df_contrats)
volumetrie.enregistrer("sinistres_bruts", df_sinistres, colonnes_sommees=("ClaimAmount",))

# Controle bloquant AVANT la jointure (EF2) : un contrat duplique fausserait
# directement la base finale, donc on arrete le traitement si on en trouve.
nb_doublons_contrats = compter_doublons(df_contrats, cle)
verifier_niveau("doublons_contrats", nb_doublons_contrats, niveaux["doublons_contrats"])

df_sinistres_orphelins = isoler_orphelins(df_sinistres, df_contrats, cle)
volumetrie.enregistrer(
    "sinistres_orphelins", df_sinistres_orphelins, colonnes_sommees=("ClaimAmount",)
)

df_base_fiabilisee = joindre_contrats_sinistres(df_contrats, df_sinistres, cle)
volumetrie.enregistrer(
    "base_fiabilisee", df_base_fiabilisee, colonnes_sommees=("montant_total_sinistres",)
)


### Volumétries avant/après (EF4)

Chaque étape a été enregistrée par `Volumetrie.enregistrer()` : nombre de
lignes et somme des montants. Sert à prouver qu'aucun contrat, sinistre ni
euro n'a disparu silencieusement (montant sinistres_bruts = montant
sinistres_orphelins + montant base_fiabilisee, aux arrondis flottants près).

In [26]:
import pandas as pd

pd.DataFrame(volumetrie.tableau())

,etape,nb_lignes,somme_ClaimAmount,somme_montant_total_sinistres
0,contrats_bruts,678013,NaN,NaN
1,sinistres_bruts,26639,6.069793e+07,NaN
2,sinistres_orphelins,195,7.887142e+05,NaN
3,base_fiabilisee,678013,NaN,59909216.5


### Écart ClaimNb (déclaré) vs nb_sinistres_reel (recalculé)

Pas de correction appliquée : on mesure seulement le nombre de contrats
concernés et le montant de charge associé, pour que le reporting puisse
expliquer l'écart plutôt que le cacher.

In [27]:
calculer_ecart_nb_sinistres(df_base_fiabilisee)

{'nb_contrats_ecart': 9117,
 'montant_concerne': 543.33,
 'nb_contrats_claimnb_sans_sinistre': 9116}

### Détail de l'écart, par valeur et avec le montant moyen associé

`calculer_ecart_nb_sinistres` ne donne qu'un total global, écrasé par les
milliers de contrats "absence totale" à 0 €. `detailler_ecart_nb_sinistres`
sépare chaque valeur d'écart (ex: -1, +1...) avec son propre montant moyen.
On affiche ensuite directement le(s) contrat(s) correspondant au cas
"vrai écart avec sinistre présent" pour le voir concrètement.

In [28]:
detailler_ecart_nb_sinistres(df_base_fiabilisee).toPandas()

,ecart_nb_sinistres,nb_contrats,montant_moyen
0,-11,1,0.000000
1,-4,2,0.000000
2,-3,20,0.000000
3,-2,485,0.000000
4,-1,8609,0.063112


Le cas "vrai écart" (pas une absence totale) : `ClaimNb` déclaré ne
correspond pas à `nb_sinistres_reel`, mais des sinistres existent bien
(`nb_sinistres_reel > 0`). On regarde directement ces contrats dans la base.

In [29]:
df_base_fiabilisee.filter(
    (F.col("ClaimNb") != F.col("nb_sinistres_reel")) & (F.col("nb_sinistres_reel") > 0)
).select(
    "IDpol", "ClaimNb", "nb_sinistres_reel", "montant_total_sinistres"
).show(truncate=False)

+-------+-------+-----------------+-----------------------+
|IDpol  |ClaimNb|nb_sinistres_reel|montant_total_sinistres|
+-------+-------+-----------------+-----------------------+
|4158255|2      |1                |543.33                 |
+-------+-------+-----------------+-----------------------+



### Hypothèse de Laurice : nb_sinistres_reel est un sous-ensemble de ClaimNb

Observation sur le seul cas de "vrai écart" (pas une absence totale) :
`ClaimNb > nb_sinistres_reel`, jamais l'inverse. Hypothèse : `ClaimNb` compte
tous les sinistres déclarés, `nb_sinistres_reel` ne compte que ceux dont le
montant est déjà connu (présents dans la table sinistres). On vérifie sur
l'ensemble des 678013 contrats, pas juste ce seul cas.

In [30]:
print("nb_sinistres_reel > ClaimNb (ne devrait jamais arriver) :",
      df_base_fiabilisee.filter(F.col("nb_sinistres_reel") > F.col("ClaimNb")).count())
print("ClaimNb > nb_sinistres_reel :",
      df_base_fiabilisee.filter(F.col("ClaimNb") > F.col("nb_sinistres_reel")).count())
print("ClaimNb == nb_sinistres_reel (coherent) :",
      df_base_fiabilisee.filter(F.col("ClaimNb") == F.col("nb_sinistres_reel")).count())

nb_sinistres_reel > ClaimNb (ne devrait jamais arriver) : 0
ClaimNb > nb_sinistres_reel : 9117
ClaimNb == nb_sinistres_reel (coherent) : 668896


### Colonne de signal : `nb_sinistres_coherent`

`joindre_contrats_sinistres` ajoute désormais cette colonne booléenne
(`ClaimNb == nb_sinistres_reel`). On ne supprime ni ne corrige aucune ligne :
les projets suivants (GLM fréquence) peuvent filtrer dessus eux-mêmes s'ils
veulent ne garder que les contrats jugés fiables.

In [31]:
df_base_fiabilisee.groupBy("nb_sinistres_coherent").count().show()

+---------------------+------+
|nb_sinistres_coherent| count|
+---------------------+------+
|                 true|668896|
|                false|  9117|
+---------------------+------+

